In [1]:
import pandas as pd
from sqlalchemy import create_engine
import numpy as np

DB_NAME = "superstore_dw"
DB_USER = "postgres"
DB_PASSWORD = "samadhi"
DB_HOST = "localhost"
DB_PORT = "5432"

connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

print("Successfully connected to the PostgreSQL database!")

Successfully connected to the PostgreSQL database!


In [2]:
df = pd.read_csv('../data/superstore.csv', encoding='latin1')
print(f"Initial row count: {len(df)}")

df = df.drop_duplicates()

df['Postal Code'] = df['Postal Code'].fillna('Unknown')
df['Postal Code'] = df['Postal Code'].astype(str).str.replace('.0', '', regex=False)

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

df['Profit_Flag'] = df['Profit'].apply(lambda x: 'Loss' if x < 0 else 'Profit')

print(f"Data cleaned. Remaining rows: {len(df)}")

Initial row count: 9994
Data cleaned. Remaining rows: 9994


In [3]:
customers = df[['Customer ID', 'Customer Name', 'Segment']].drop_duplicates(subset=['Customer ID'])

customers = customers.rename(columns={
    'Customer ID': 'customerid',
    'Customer Name': 'customername',
    'Segment': 'segment'
})

customers.to_sql('dim_customer', engine, if_exists='append', index=False, method='multi')
print(f" Loaded {len(customers)} unique customers into dim_customer")

 Loaded 793 unique customers into dim_customer


In [4]:
products = df[['Product ID', 'Category', 'Sub-Category', 'Product Name']].drop_duplicates(subset=['Product ID'])

products = products.rename(columns={
    'Product ID': 'productid',
    'Category': 'category',
    'Sub-Category': 'subcategory',
    'Product Name': 'productname'
})

products.to_sql('dim_product', engine, if_exists='append', index=False, method='multi')
print(f"Loaded {len(products)} unique products into dim_product")

Loaded 1862 unique products into dim_product


In [5]:
orders = df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 
             'Region', 'State', 'City', 'Postal Code', 'Country']].drop_duplicates(subset=['Order ID'])

orders = orders.rename(columns={
    'Order ID': 'orderid',
    'Order Date': 'orderdate',
    'Ship Date': 'shipdate',
    'Ship Mode': 'shipmode',
    'Region': 'region',
    'State': 'state',
    'City': 'city',
    'Postal Code': 'postalcode',
    'Country': 'country'
})

orders.to_sql('dim_order', engine, if_exists='append', index=False, method='multi')
print(f"Loaded {len(orders)} unique orders into DIM_ORDER")

Loaded 5009 unique orders into DIM_ORDER


In [6]:
all_dates = pd.concat([df['Order Date'], df['Ship Date']]).dropna().unique()
date_df = pd.DataFrame({'fulldate': pd.to_datetime(all_dates)})

date_df['timekey'] = date_df['fulldate'].dt.strftime('%Y%m%d').astype(int)
date_df['year'] = date_df['fulldate'].dt.year
date_df['quarter'] = date_df['fulldate'].dt.quarter
date_df['month'] = date_df['fulldate'].dt.month
date_df['monthname'] = date_df['fulldate'].dt.strftime('%B') # Full month name
date_df['dayofweek'] = date_df['fulldate'].dt.dayofweek
date_df['dayname'] = date_df['fulldate'].dt.day_name()
date_df['weekofyear'] = date_df['fulldate'].dt.isocalendar().week
date_df['isweekend'] = date_df['dayofweek'].isin([5, 6])

date_df.to_sql('dim_time', engine, if_exists='append', index=False, method='multi')
print(f"Loaded {len(date_df)} unique dates into DIM_TIME")

Loaded 1434 unique dates into DIM_TIME


In [7]:
customer_map = pd.read_sql("SELECT customerkey, customerid FROM dim_customer", engine)
product_map = pd.read_sql("SELECT productkey, productid FROM dim_product", engine)
order_map = pd.read_sql("SELECT orderkey, orderid FROM dim_order", engine)

df['DateKey'] = df['Order Date'].dt.strftime('%Y%m%d').astype(int)

fact = df.merge(customer_map, left_on='Customer ID', right_on='customerid', how='inner')
fact = fact.merge(product_map, left_on='Product ID', right_on='productid', how='inner')
fact = fact.merge(order_map, left_on='Order ID', right_on='orderid', how='inner')

fact_final = fact[['customerkey', 'productkey', 'orderkey', 'DateKey', 
                   'Order ID', 'Sales', 'Quantity', 'Discount', 'Profit']].copy()

fact_final = fact_final.rename(columns={
    'DateKey': 'timekey',
    'Order ID': 'orderid',
    'Sales': 'sales',
    'Quantity': 'quantity',
    'Discount': 'discount',
    'Profit': 'profit'
})

fact_final.to_sql('fact_sales', engine, if_exists='append', index=False, method='multi')
print(f"Loaded {len(fact_final)} transactions into FACT_SALES")

Loaded 9994 transactions into FACT_SALES


In [8]:
row_count = pd.read_sql("SELECT COUNT(*) as total_rows FROM FACT_SALES", engine)
print("\n--- FINAL DATA WAREHOUSE VALIDATION ---")
print(f"Total rows in FACT_SALES: {row_count['total_rows'][0]}")

print("\nFirst 5 rows of FACT_SALES:")
display(pd.read_sql("SELECT * FROM FACT_SALES LIMIT 5", engine))


--- FINAL DATA WAREHOUSE VALIDATION ---
Total rows in FACT_SALES: 9994

First 5 rows of FACT_SALES:


,saleskey,customerkey,productkey,orderkey,timekey,orderid,sales,quantity,discount,profit
0,1,1,1,1,20161108,CA-2016-152156,261.96,2,0.0,41.91
1,2,1,2,1,20161108,CA-2016-152156,731.94,3,0.0,219.58
2,3,152,1,4668,20170501,CA-2017-110198,314.35,3,0.2,-15.72
3,4,152,1569,4668,20170501,CA-2017-110198,4.61,2,0.2,1.50
4,5,278,1,4301,20170825,CA-2017-159793,130.98,2,0.5,-89.07
